In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train = CIFAR10(root='./Data', train=True, download=True, transform=transforms)
test = CIFAR10(root='./Data', train=False, download=True, transform=transforms)

100%|██████████| 170M/170M [01:05<00:00, 2.61MB/s]


Building the CNN


In [15]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )
    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )
  def forward(self,x):
      x = self.conv_layers(x)
      x = x.view(x.size(0),-1)
      x = self.fc_layers(x)
      return x

In [16]:
model = CNN()

In [17]:
 criterion = nn.CrossEntropyLoss()
 optimiser = optim.Adam(model.parameters())

Training CNN

In [18]:
trainloader = DataLoader(train, batch_size=32, shuffle=True, num_workers=2)
testloader = DataLoader(test, batch_size=32, shuffle=False, num_workers=2)

In [20]:
epochs = 10
for epoch in range(epochs):
  epoch_training_loss = 0.0
  for images, labels in trainloader:
    optimiser.zero_grad()
    output = model.forward(images)
    loss = criterion(output,labels)
    loss.backward()
    optimiser.step()
    epoch_training_loss = loss.item()
  print(f"epoch = {epoch} , loss = {epoch_training_loss}")

epoch = 0 , loss = 0.9460155963897705
epoch = 1 , loss = 1.2141039371490479
epoch = 2 , loss = 0.6241957545280457
epoch = 3 , loss = 0.36514219641685486
epoch = 4 , loss = 0.4892958104610443
epoch = 5 , loss = 0.7797060012817383
epoch = 6 , loss = 0.5520654320716858
epoch = 7 , loss = 0.19589863717556
epoch = 8 , loss = 0.26891177892684937
epoch = 9 , loss = 0.14645938575267792


In [23]:
#evaluate
correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
  for images,labels in testloader:
    outputs = model.forward(images)
    _, predicted = torch.max(outputs,1)
    correct_labels += (predicted == labels).sum().item()
    total_labels+=labels.size(0)
print(f"accuracy = {correct_labels/total_labels*100}")

accuracy = 74.57000000000001
